In [1]:
from dotenv import load_dotenv
import os
from pathlib import Path
from openai import AzureOpenAI
from embeddings import UserDocumentCollection
import os, json, re
import pandas as pd

import numpy as np
from pypdf import PdfReader
import win32com.client as win32
from time import sleep

In [2]:
load_dotenv()

True

In [3]:
folder_path = os.environ.get("FOLDER_PATH")

## Position papers

In [4]:
docs_path = Path(folder_path) / "position-papers"
output_folder = Path(folder_path).parent / "results"

In [5]:
container_name="sante-docs"
colect = UserDocumentCollection(container_name)


# iterate and upload all files in docs_path
for p in sorted(Path(docs_path).iterdir()):
    if not p.is_file():
        continue
    with p.open("rb") as file_obj:
        colect.add_file_to_blob_container(str(p), file_obj)

colect.create_user_search_index()
colect.create_data_source_connection()
colect.create_user_skillset()
colect.create_user_indexer()
colect.reset_and_run_indexer()

2025-11-03 10:45:52.259 | INFO     | embeddings:get_container:66 - Container 'sante-docs' already exists.
2025-11-03 10:45:52.350 | WARNING  | embeddings:add_file_to_blob_container:99 - File 'F3552803-Feedback Paper On the European Biotech Act.pdf' already exists in container 'sante-docs'. Skipping upload.
2025-11-03 10:45:52.363 | WARNING  | embeddings:add_file_to_blob_container:99 - File 'F3553888-cpme.2025-092.FINAL.Statement.Biotech.Act.pdf' already exists in container 'sante-docs'. Skipping upload.
2025-11-03 10:45:52.376 | WARNING  | embeddings:add_file_to_blob_container:99 - File 'F3554271-Koppert submission for the biotech act consultations.pdf' already exists in container 'sante-docs'. Skipping upload.
2025-11-03 10:45:52.393 | WARNING  | embeddings:add_file_to_blob_container:99 - File 'F3559517-EBSA Position Paper - OMNIBUS - 20250416 (1).pdf' already exists in container 'sante-docs'. Skipping upload.
2025-11-03 10:45:52.422 | WARNING  | embeddings:add_file_to_blob_container:

In [9]:
colect.reset_and_run_indexer()

2025-11-03 10:10:03.341 | INFO     | embeddings:reset_and_run_indexer:252 - 🧹 Indexer 'sante-docs-indexer' checkpoint reset successfully.
2025-11-03 10:10:03.495 | INFO     | embeddings:reset_and_run_indexer:256 - 🚀 Indexer 'sante-docs-indexer' started successfully.


In [5]:
BRUSSEL_first_PROMPT = (
  """

  Below are concise definitions of three key topics related to biotechnology policy and industry development:

  - ATMP (Advanced Therapy Medicinal Products): Medicinal products based on genes, cells, or engineered tissues offering advanced or personalised therapeutic effects (e.g. gene therapy, CAR-T cell therapy, tissue-engineered grafts). They often overlap with personalised and regenerative medicine, requiring complex regulatory and manufacturing pathways.
      • gene_therapies: Advanced therapy medicinal producsts based on genes
      • cell_therapies: Advanced therapy medicinal products based on cells
      • tissue_engineered_products: Advanced therapy medicinal products based on engineered tissues
      • personalised_medicine: Advanced therapy medicinal products tailored to personalised therapeutic effects
  - Access to Markets: Factors affecting how biotech products reach patients or users, including:
      • access_hurdles — document criticizes the regulatory, reimbursement, validation, or manufacturing obstacles delaying market entry;
      • access_strategies — document promotes, supports, encourages policy or business approaches to address these barriers (e.g. harmonisation, early regulatory dialogue, public procurement, risk-sharing models).

  - VC_Investments_IPO (Venture Capital, Investments, and Stock Exchanges/IPOs): Trends and mechanisms of biotech financing, including venture capital, private equity, institutional investments, and IPOs. This includes availability of capital, exit routes, market maturity, and cross-border funding.

  TASK:
  For each theme and subtheme, check whether the document makes a reference to it.
  If yes, return aLL exact citations and page numbers.
  If not present, set the value to null.

  Output format:
  Return ONE JSON object with EXACTLY these keys; each value is EITHER null OR a list of strings containing exact text excerpts in the format “... (page XX)”:
  {
    "ATMP": <null or ["<exact quote> (page XX)", "<exact quote> (page XX)", ...]>,
    "ATMP_gene_therapies": <null or ["<exact quote> (page XX)", "<exact quote> (page XX)", ...]>,
    "ATMP_cell_therapies": <null or ["<exact quote> (page XX)", "<exact quote> (page XX)", ...]>,
    "ATMP_tissue_engineered_products": <null or ["<exact quote> (page XX)", "<exact quote> (page XX)", ...]>,
    "ATMP_personalised_medicine": <null or ["<exact quote> (page XX)", "<exact quote> (page XX)", ...]>,
    "Access_to_Markets_hurdles": <null or ["<exact quote> (page XX)", "<exact quote> (page XX)", ...]>,
    "Access_to_Markets_strategies": <null or ["<exact quote> (page XX)", "<exact quote> (page XX)", ...]>,
    "VC_Investments_IPO": <null or ["<exact quote> (page XX)", "<exact quote> (page XX)", ...]>
  }

  Rules:
  - Base your answer ONLY on the provided document content.
  - Use **exact quotations** and page numbers where the reference occurs. Do not return extract IDs like [doc1], [doc2], etc.
  - Each citation must be a **full, grammatically complete sentence** exactly as it appears in the document, ending with proper punctuation (e.g. period, semicolon, or question mark) before the page reference.
  - Do not return partial phrases, ellipses (...), or truncated text.
  - Do not return "the document mentions...", instead return the actual citation text.
  - Do NOT invent or infer information; if uncertain or if information is not available, use null.
  - Do not insert arbitrary newline characters in the quotes.
  - Output VALID JSON and NOTHING ELSE (no markdown, no prose, no commentary).
  """
)

In [6]:
client = AzureOpenAI(
    azure_endpoint=os.environ["AZURE_COGNITIVE_SERVICES_ENDPOINT"],
    api_key=os.environ["AZURE_COGNITIVE_API"],
    api_version="2024-10-21",  # 2024-02-01+ supports data_sources
)

In [7]:
def odata_escape(s: str) -> str:
    # OData string literals escape single quotes by doubling them
    return s.replace("'", "''")

In [8]:
def first_information(doc_name):
    completion = client.chat.completions.create(
        model="gpt-model",
        messages=[
            {
                "role": "user",
                "content": f"You already possess the document {doc_name} that contains all necessary information to answer following questions.\n"
                           + BRUSSEL_first_PROMPT
            }
        ],
        response_format={"type": "json_object"},
        extra_body={
            "data_sources": [
                {
                    "type": "azure_search",
                    "parameters": {
                        "endpoint": os.environ["AZURE_AI_SEARCH_ENDPOINT"],
                        "index_name": f"{container_name}_index",
                        "authentication": {
                            "type": "api_key",
                            "key": os.environ["AZURE_AI_SEARCH_API_KEY"],
                        },
                        "fields_mapping": {
                            "title_field": "title",
                            "filepath_field": "filepath",
                            "url_field": "url",
                        },
                        "filter": f"title eq '{odata_escape(doc_name)}'",
                        "in_scope": True,
                    },
                }
            ],
        },
    )

    raw = completion.choices[0].message.content or ""
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        # Fallback schema matching the new prompt keys
        return {
            "ATMP": None,
            "ATMP_gene_therapies": None,
            "ATMP_cell_therapies": None,
            "ATMP_tissue_engineered_products": None,
            "ATMP_personalised_medicine": None,
            "Access_to_Markets_hurdles": None,
            "Access_to_Markets_strategies": None,
            "VC_Investments_IPO": None,
        }


In [46]:
docs = sorted([p.name for p in Path(docs_path).iterdir() if p.is_file()])[11:15]

for doc in docs:
    info = first_information(doc)

    print("=" * 80)
    print(f"📄 Document: {doc}")
    print("-" * 80)
    print(json.dumps(info, indent=2, ensure_ascii=False))
    print("=" * 80 + "\n")

📄 Document: F3563929-CFE response to Biotech Act proposal.pdf
--------------------------------------------------------------------------------
{
  "ATMP": null,
  "ATMP_gene_therapies": null,
  "ATMP_cell_therapies": null,
  "ATMP_tissue_engineered_products": null,
  "ATMP_personalised_medicine": null,
  "Access_to_Markets_hurdles": [
    "The proposal acknowledges that the complex regulatory framework of the EU single market, which may diverge among Member States, is perceived to be slow and burdensome. In a rapidly growing and evolving sector such as biotech, such frameworks can act as a significant barrier to progress. (page 1)",
    "This is particularly pertinent for biotechnologies for non-animal science that, while proving promising in the research and development phase, can face several hurdles when it comes to regulatory uptake, for example slow or lack of validation or low recognition of regulatory utility. (page 1)"
  ],
  "Access_to_Markets_strategies": [
    "As such, the 

### SECOND PROMPT

In [9]:
BRUSSEL_SECOND_PROMPT = (
  """

  Below are concise definitions of three key topics related to biotechnology policy and industry development:

  - Manufacturing Capabilities: Refers to the existence and advancement of state-of-the-art infrastructure, equipment, and technologies used in biotechnology manufacturing. This includes facilities and processes for bioproduction, bioprocess optimization, automation, and quality systems meeting regulatory (e.g., GMP) standards for scale-up and commercialization.

  - Incubation and Acceleration: Structured programs that support early-stage biotech startups. 
      • Incubation refers to very early-stage support, typically offering laboratory space, mentoring, and assistance in building viable companies.  
      • Acceleration focuses on more mature startups, providing investment readiness, business development, and market access.  
    Identify any references to such programs, initiatives, or national policies supporting biotech incubation or acceleration.

  - Talent Retention: Refers to policies and measures to retain qualified biotech professionals and prevent talent loss. Includes references to career development, research funding, competitive compensation, and efforts to prevent brain drain or migration of expertise.

  - Upskilling / Reskilling: Biomanufacturing requires deep process expertise (e.g., fermentation, molecular methods) alongside cross-cutting skills in data/AI, sustainability, systems thinking, and entrepreneurship. This theme refers to education and training initiatives aimed at improving or renewing workers’ skills to meet evolving biotechnology needs, including continuous learning, professional training, workforce transition, and capacity-building programs.


  TASK:
  For each theme, check whether the document makes a substantive reference to it.
  If present, return ONLY exact quotations from the document with their page numbers.
  Do NOT summarize, paraphrase, or interpret. If not present, set the value to null.

  Output format:
  Return ONE JSON object with EXACTLY these keys; each value is EITHER null OR a list of strings containing exact text excerpts in the format “... (page XX)”:
    {
      "Manufacturing_Capabilities": <null or ["<exact quote> (page XX)", "<exact quote> (page XX)", ...]>,
      "Incubation_and_Acceleration": <null or ["<exact quote> (page XX)", "<exact quote> (page XX)", ...]>,
      "Talent_Retention": <null or ["<exact quote> (page XX)", "<exact quote> (page XX)", ...]>,
      "Upskilling_Reskilling": <null or ["<exact quote> (page XX)", "<exact quote> (page XX)", ...]>
    }

  Rules:
  - Base your answer ONLY on the provided document content.
  - Use **exact citations and page numbers** where the reference occurs. Do not return extract id like [doc1], [doc2], etc.
  - Each citation must be a **full, grammatically complete sentence** exactly as it appears in the document, ending with proper punctuation (e.g. period, semicolon, or question mark) before the page reference.
  - Do not return partial phrases, ellipses (...), or truncated text.
  - Do not return "the document mentions...", instead return the actual citation text.
  - Do not insert arbitrary newline characters in the quotes.
  - If a topic appears on multiple pages, include multiple citation objects.
  - Do NOT invent or infer information; if uncertain or if information is not available, use null.
  - Output VALID JSON and NOTHING ELSE (no markdown, no prose, no commentary).
  """
)




In [10]:
def second_information(doc_name):
    completion = client.chat.completions.create(
        model="gpt-model",
        messages=[
            {
                "role": "user",
                "content": f"You already possess the document {doc_name} that contains all necessary information to answer following questions.\n"
                           + BRUSSEL_SECOND_PROMPT
            }
        ],
        response_format={"type": "json_object"},
        extra_body={
            "data_sources": [
                {
                    "type": "azure_search",
                    "parameters": {
                        "endpoint": os.environ["AZURE_AI_SEARCH_ENDPOINT"],
                        "index_name": f"{container_name}_index",
                        "authentication": {
                            "type": "api_key",
                            "key": os.environ["AZURE_AI_SEARCH_API_KEY"],
                        },
                        "fields_mapping": {
                            "title_field": "title",
                            "filepath_field": "filepath",
                            "url_field": "url",
                        },
                        "filter": f"title eq '{odata_escape(doc_name)}'",
                        "in_scope": True,
                    },
                }
            ],
        },
    )

    raw = completion.choices[0].message.content or ""
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        # Fallback schema matching updated BRUSSEL_SECOND_PROMPT keys
        return {
            "Manufacturing_Capabilities": None,
            "Incubation_and_Acceleration": None,
            "Talent_Retention": None,
            "Upskilling_Reskilling": None,
        }



In [49]:
docs = sorted([p.name for p in Path(docs_path).iterdir() if p.is_file()])[:10]

for doc in docs:
    info = second_information(doc)

    print("=" * 80)
    print(f"📄 Document: {doc}")
    print("-" * 80)
    print(json.dumps(info, indent=2, ensure_ascii=False))
    print("=" * 80 + "\n")

📄 Document: F3552803-Feedback Paper On the European Biotech Act.pdf
--------------------------------------------------------------------------------
{
  "Manufacturing_Capabilities": [
    "While the EU is a leader in biologics manufacturing, it remains vulnerable to geopolitical supply chain risks and foreign investment outflows. Dual-use biotech technologies in health and agriculture further raise security concerns. (page 6)",
    "Launch a Biomanufacturing Sovereignty Fund to support onshore production capacity for APIs, enzymes, and cell-based products. Simplify permitting and create industrial biozones with dedicated infrastructure and environmental oversight.. (page 7)"
  ],
  "Incubation_and_Acceleration": null,
  "Workforce_Development": [
    "A mismatch exists between the labor market and biotech's evolving skills needs. Europe's biotech hubs suffer from a shortage of bioprocess engineers, data scientists, regulatory affairs experts, and translational researchers. (page 5)",


BadRequestError: Error code: 400 - {'error': {'requestid': '02fe0226-605c-4e47-82e9-cef3583235c9', 'code': 400, 'message': 'An error occurred when calling Azure OpenAI: Server responded with status 429. Error message: {"error":{"code":"429","message": "Rate limit is exceeded. Try again in 7 seconds."}}'}}

### tHIRD PROMPT

In [11]:
BRUSSEL_THIRD_PROMPT = (
  """

  Below are concise definitions of three key topics related to biotechnology policy and industry development:

  - Artificial Intelligence (AI) related Challenges in Biotechnology: Refers to the t'echnical, regulatory or ethical obstacles, limitations and bottlenecks that constrain the responsible use of AI in biotechnology.

  - Artificial Intelligence (AI) Development and Deployment in Biotechnology: Refers to cases where the document promotes, supports, or encourages the practical use, integration, or scaling of AI technologies in biotechnology. This includes references to initiatives, investments, strategies, or policies aimed at developing or deploying AI tools in biotech research, product development, or manufacturing.

  - Global Collaboration: Refers to international partnerships, initiatives, or cooperative frameworks between European biotechnology actors and global counterparts.

  - Biosecurity: Refers to the protection, control, and accountability of biological materials, data, and technologies to prevent their unauthorized access, loss, theft, misuse, diversion, or intentional release.  
      • Nucleic Acid Synthesis Screening — A specific biosecurity measure involving the screening of DNA or RNA synthesis requests to prevent the production of pathogenic or otherwise harmful sequences.

  TASK:
  For each theme, check whether the document makes a substantive reference to it.
  If present, return ONLY exact quotations from the document with their page numbers.
  Do NOT summarize, paraphrase, or interpret. If not present, set the value to null.

  Output format:
  Return ONE JSON object with EXACTLY these keys; each value is EITHER null OR a list of strings containing exact text excerpts in the format “... (page XX)”:
  {
    "AI_Challenges": <null or ["<exact quote> (page XX)", "<exact quote> (page XX)", ...]>,
    "AI_Development_and_Deployment": <null or ["<exact quote> (page XX)", "<exact quote> (page XX)", ...]>,
    "Global_Collaboration": <null or ["<exact quote> (page XX)", "<exact quote> (page XX)", ...]>,
    "Biosecurity": <null or ["<exact quote> (page XX)", "<exact quote> (page XX)", ...]>,
    "Biosecurity_Nucleic_Acid_Screening": <null or ["<exact quote> (page XX)", "<exact quote> (page XX)", ...]>
  }

  Rules:
  - Base your answer ONLY on the provided document content.
  - Use **exact citations and page numbers** where the reference occurs. Do not return extract IDs like [doc1], [doc2], etc.
  - Each citation must be a **full, grammatically complete sentence** exactly as it appears in the document, ending with proper punctuation (e.g. period, semicolon, or question mark) before the page reference.
  - Do not return partial phrases, ellipses (...), or truncated text.
  - Do not return "the document mentions...", instead return the actual citation text.
  - Do not insert arbitrary newline characters in the quotes.
  - If a topic appears on multiple pages, include multiple citation strings.
  - Do NOT invent or infer information; if uncertain or if information is not available, use null.
  - Output VALID JSON and NOTHING ELSE (no markdown, no prose, no commentary).
  """
)


In [12]:
def third_information(doc_name):
    completion = client.chat.completions.create(
        model="gpt-model",
        messages=[
            {
                "role": "user",
                "content": f"You already possess the document {doc_name} that contains all necessary information to answer following questions.\n"
                           + BRUSSEL_THIRD_PROMPT
            }
        ],
        response_format={"type": "json_object"},
        extra_body={
            "data_sources": [
                {
                    "type": "azure_search",
                    "parameters": {
                        "endpoint": os.environ["AZURE_AI_SEARCH_ENDPOINT"],
                        "index_name": f"{container_name}_index",
                        "authentication": {
                            "type": "api_key",
                            "key": os.environ["AZURE_AI_SEARCH_API_KEY"],
                        },
                        "fields_mapping": {
                            "title_field": "title",
                            "filepath_field": "filepath",
                            "url_field": "url",
                        },
                        "filter": f"title eq '{odata_escape(doc_name)}'",
                        "in_scope": True,
                    },
                }
            ],
        },
    )

    raw = completion.choices[0].message.content or ""
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        # Fallback schema matching BRUSSEL_THIRD_PROMPT keys
        return {
            "AI_Challenges": None,
            "AI_Development_and_Deployment": None,
            "Global_Collaboration": None,
            "Biosecurity": None,
            "Biosecurity_Nucleic_Acid_Screening": None,
        }


In [ ]:
docs = sorted([p.name for p in Path(docs_path).iterdir() if p.is_file()])[:10]

for doc in docs:
    info = third_information(doc)

    print("=" * 80)
    print(f"📄 Document: {doc}")
    print("-" * 80)
    print(json.dumps(info, indent=2, ensure_ascii=False))
    print("=" * 80 + "\n")

📄 Document: F3552803-Feedback Paper On the European Biotech Act.pdf
--------------------------------------------------------------------------------
{
  "AI_Challenges": [
    "The biotech revolution depends increasingly on access to real-world data, AI-enhanced simulations, and supercomputing resources. However, many biotech SMEs lack access to the European Health Data Space (EHDS), federated biobanks, or AI testing environments. Academic literature highlights the role of machine learning in expediting drug discovery, protein folding, and metabolic pathway design (Reis et al., 2020). The Biotech Act must integrate the EU AI Strategy and EHDS with dedicated capacity for biotech applications. (page 5)",
    "Ensure biotech-specific access to the EHDS, federated datasets, and AI Factories. Mandate interoperability standards and ethical safeguards. Support the creation of an EU Biotech-AI Alliance to encourage cross-sector R&D. (page 7)"
  ],
  "AI_Development_and_Deployment": [
    "The 

## ALL PROMPTS

In [23]:

from time import sleep, strftime
from openai import APIConnectionError, BadRequestError
import re

def call_once_with_wait(fn, *args, delay=10, label=None, **kwargs):
    who = f"[{label}]" if label else ""
    print(f"{strftime('%H:%M:%S')} {who} waiting {delay}s before call...", flush=True)
    sleep(delay)
    try:
        out = fn(*args, **kwargs)
        print(f"{strftime('%H:%M:%S')} {who} call OK", flush=True)
        return out
    except BadRequestError as e:
        s = str(e)
        if "Rate limit is exceeded" in s:
            m = re.search(r"Try again in\s+(\d+)\s+seconds?", s, re.I)
            wait_s = int(m.group(1)) + 1 if m else max(5, delay)
            print(f"{strftime('%H:%M:%S')} {who} 429 rate limit → waiting {wait_s}s, then retrying once...", flush=True)
            sleep(wait_s)
            out = fn(*args, **kwargs)
            print(f"{strftime('%H:%M:%S')} {who} retry OK", flush=True)
            return out
        print(f"{strftime('%H:%M:%S')} {who} BadRequestError (not rate-limit): {e}", flush=True)
        raise
    except APIConnectionError as e:
        print(f"{strftime('%H:%M:%S')} {who} connection error → waiting 5s, then retrying once... ({e})", flush=True)
        sleep(5)
        out = fn(*args, **kwargs)
        print(f"{strftime('%H:%M:%S')} {who} retry OK", flush=True)
        return out
    except Exception as e:
        print(f"{strftime('%H:%M:%S')} {who} unexpected error: {e}", flush=True)
        raise


In [28]:
from tqdm.auto import tqdm
from pathlib import Path
import json
import pandas as pd
import time

# --- parameters ---
SAVE_EVERY = 10                     # how often to save (every N docs)
OUTPUT_PATH = Path(output_folder) / "brussels_pp_results.json"


docs = sorted([p.name for p in docs_path.iterdir() if p.is_file()])

results, errors = [], []
pbar = tqdm(docs, desc="Processing documents", unit="doc", total=len(docs))

def save_checkpoint():
    """Save current results and errors to disk."""
    try:
        # timestamped checkpoint
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        temp_file = OUTPUT_PATH.with_name(f"{OUTPUT_PATH.stem}_checkpoint_{timestamp}.json")

        with open(temp_file, "w", encoding="utf-8") as f:
            json.dump({"results": results, "errors": errors}, f, ensure_ascii=False, indent=2)
        pbar.write(f"[SAVED] checkpoint with {len(results)} results → {temp_file}")
    except Exception as e:
        pbar.write(f"[WARNING] Failed to save checkpoint: {e}")

for i, doc in enumerate(pbar, start=1):
    pbar.set_postfix_str(doc)
    try:
        first  = call_once_with_wait(first_information, doc, delay=10, label=f"first:{doc}")
        second = call_once_with_wait(second_information, doc, delay=10, label=f"second:{doc}")
        third  = call_once_with_wait(third_information, doc, delay=10, label=f"third:{doc}")
        results.append({**first, **second, **third})
    except Exception as e:
        errors.append((doc, repr(e)))
        pbar.write(f"[ERROR] {doc}: {e}")

    # --- periodic save ---
    if i % SAVE_EVERY == 0:
        save_checkpoint()

# --- final save ---
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump({"results": results, "errors": errors}, f, ensure_ascii=False, indent=2)
pbar.write(f"[DONE] All results saved to {OUTPUT_PATH}")


Processing documents:   0%|          | 0/149 [00:00<?, ?doc/s, F3552803-Feedback Paper On the European Biotech Act.pdf]

13:30:44 [first:F3552803-Feedback Paper On the European Biotech Act.pdf] waiting 10s before call...


13:31:00 [first:F3552803-Feedback Paper On the European Biotech Act.pdf] call OK
13:31:00 [second:F3552803-Feedback Paper On the European Biotech Act.pdf] waiting 10s before call...
13:31:14 [second:F3552803-Feedback Paper On the European Biotech Act.pdf] call OK
13:31:14 [third:F3552803-Feedback Paper On the European Biotech Act.pdf] waiting 10s before call...
13:31:31 [third:F3552803-Feedback Paper On the European Biotech Act.pdf] call OK


Processing documents:   1%|          | 1/149 [00:47<1:56:08, 47.08s/doc, F3553888-cpme.2025-092.FINAL.Statement.Biotech.Act.pdf] 

13:31:31 [first:F3553888-cpme.2025-092.FINAL.Statement.Biotech.Act.pdf] waiting 10s before call...
13:31:43 [first:F3553888-cpme.2025-092.FINAL.Statement.Biotech.Act.pdf] call OK
13:31:43 [second:F3553888-cpme.2025-092.FINAL.Statement.Biotech.Act.pdf] waiting 10s before call...
13:31:55 [second:F3553888-cpme.2025-092.FINAL.Statement.Biotech.Act.pdf] call OK
13:31:55 [third:F3553888-cpme.2025-092.FINAL.Statement.Biotech.Act.pdf] waiting 10s before call...
13:32:23 [third:F3553888-cpme.2025-092.FINAL.Statement.Biotech.Act.pdf] call OK


...

15:18:07 [first:F3565532-BiotechAct_VPHi.pdf] waiting 10s before call...
15:18:20 [first:F3565532-BiotechAct_VPHi.pdf] call OK
15:18:20 [second:F3565532-BiotechAct_VPHi.pdf] waiting 10s before call...
15:18:33 [second:F3565532-BiotechAct_VPHi.pdf] call OK
15:18:33 [third:F3565532-BiotechAct_VPHi.pdf] waiting 10s before call...
15:18:46 [third:F3565532-BiotechAct_VPHi.pdf] call OK


Processing documents:  99%|█████████▉| 148/149 [1:48:02<00:43, 43.18s/doc, F3582124-Q12_cfe_biotech_act_Fondazione Telethon.docx]

15:18:46 [first:F3582124-Q12_cfe_biotech_act_Fondazione Telethon.docx] waiting 10s before call...
15:19:05 [first:F3582124-Q12_cfe_biotech_act_Fondazione Telethon.docx] call OK
15:19:05 [second:F3582124-Q12_cfe_biotech_act_Fondazione Telethon.docx] waiting 10s before call...
15:19:19 [second:F3582124-Q12_cfe_biotech_act_Fondazione Telethon.docx] call OK
15:19:19 [third:F3582124-Q12_cfe_biotech_act_Fondazione Telethon.docx] waiting 10s before call...
15:19:31 [third:F3582124-Q12_cfe_biotech_act_Fondazione Telethon.docx] call OK


Processing documents: 100%|██████████| 149/149 [1:48:47<00:00, 43.81s/doc, F3582124-Q12_cfe_biotech_act_Fondazione Telethon.docx]

[DONE] All results saved to C:\Users\YağmurAslan\OneDrive - Technopolis Group Ltd\RAG_Brussels\results\brussels_pp_results.json


In [29]:
results[:5]

[{'ATMP': ['Biotech firms report difficulty navigating EU regulatory environments, particularly for clinical trials, GMOs, and advanced therapies. (page 4)',
   'The European Biotech Act was introduced in response to these challenges and is expected to form the legislative backbone of a broader European life sciences strategy. It seeks to simplify complex regulatory pathways, streamline product development pipelines, and harmonize fragmented biotech policies across Member States. (page 4)'],
  'ATMP_gene_therapies': None,
  'ATMP_cell_therapies': None,
  'ATMP_tissue_engineered_products': None,
  'ATMP_personalised_medicine': None,
  'Access_to_Markets_hurdles': ['Biotech firms report difficulty navigating EU regulatory environments, particularly for clinical trials, GMOs, and advanced therapies. Regulatory divergence across Member States results in unnecessary costs, slow time-to-market, and underutilized research findings. (page 4)',
   'The EU biotech investment landscape is dominat

In [30]:
FINAL_JSON_PATH = Path(output_folder) / "brussels_final_results.json"

try:
    # dump your list of dictionaries (results) as a single JSON array
    with open(FINAL_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"[DONE] Final merged results saved to {FINAL_JSON_PATH}")
except Exception as e:
    print(f"[ERROR] Failed to save final JSON: {e}")

[DONE] Final merged results saved to C:\Users\YağmurAslan\OneDrive - Technopolis Group Ltd\RAG_Brussels\results\brussels_final_results.json


In [31]:
import pandas as pd
from pathlib import Path

def _tolist(x):
    if isinstance(x, list):
        return x
    return [x] if x else []

# Align results to docs (safe even if lengths differ)
rows = []
for doc_name, r in zip(docs, results):
    title = Path(doc_name).stem  # filename without extension

    rows.append({
        "title": title,  # <-- first column from docs
        # --- First group (BRUSSEL_FIRST_PROMPT) ---
        "ATMP": _tolist(r.get("ATMP")),
        "ATMP_gene_therapies": _tolist(r.get("ATMP_gene_therapies")),
        "ATMP_cell_therapies": _tolist(r.get("ATMP_cell_therapies")),
        "ATMP_tissue_engineered_products": _tolist(r.get("ATMP_tissue_engineered_products")),
        "ATMP_personalised_medicine": _tolist(r.get("ATMP_personalised_medicine")),
        "Access_to_Markets_hurdles": _tolist(r.get("Access_to_Markets_hurdles")),
        "Access_to_Markets_strategies": _tolist(r.get("Access_to_Markets_strategies")),
        "VC_Investments_IPO": _tolist(r.get("VC_Investments_IPO")),

        # --- Second group (BRUSSEL_SECOND_PROMPT) ---
        "Manufacturing_Capabilities": _tolist(r.get("Manufacturing_Capabilities")),
        "Incubation_and_Acceleration": _tolist(r.get("Incubation_and_Acceleration")),
        "Workforce_Development": _tolist(r.get("Workforce_Development")),

        # --- Third group (BRUSSEL_THIRD_PROMPT) ---
        "AI_Challenges": _tolist(r.get("AI_Challenges")),
        "AI_Development_and_Deployment": _tolist(r.get("AI_Development_and_Deployment")),
        "Global_Collaboration": _tolist(r.get("Global_Collaboration")),
        "Biosecurity": _tolist(r.get("Biosecurity")),
        "Biosecurity_Nucleic_Acid_Screening": _tolist(r.get("Biosecurity_Nucleic_Acid_Screening")),
    })

df = pd.DataFrame(
    rows,
    columns=[
        "title",  # first column
        "ATMP", "ATMP_gene_therapies", "ATMP_cell_therapies",
        "ATMP_tissue_engineered_products", "ATMP_personalised_medicine",
        "Access_to_Markets_hurdles", "Access_to_Markets_strategies", "VC_Investments_IPO",
        "Manufacturing_Capabilities", "Incubation_and_Acceleration", "Workforce_Development",
        "AI_Challenges", "AI_Development_and_Deployment", "Global_Collaboration",
        "Biosecurity", "Biosecurity_Nucleic_Acid_Screening"
    ]
)


In [32]:
# Split the existing 'title' column into two new columns: 'reference' and 'title'
df[["reference", "title"]] = df["title"].str.split("-", n=1, expand=True)

# Strip whitespace
df["reference"] = df["reference"].str.strip()
df["title"] = df["title"].str.strip()

# Move 'reference' to be the first column
cols = ["reference"] + [c for c in df.columns if c != "reference"]
df = df[cols]
df.head()

,reference,title,ATMP,ATMP_gene_therapies,ATMP_cell_therapies,ATMP_tissue_engineered_products,ATMP_personalised_medicine,Access_to_Markets_hurdles,Access_to_Markets_strategies,VC_Investments_IPO,Manufacturing_Capabilities,Incubation_and_Acceleration,Workforce_Development,AI_Challenges,AI_Development_and_Deployment,Global_Collaboration,Biosecurity,Biosecurity_Nucleic_Acid_Screening
0,F3552803,Feedback Paper On the European Biotech Act,[Biotech firms report difficulty navigating EU...,[],[],[],[],[Biotech firms report difficulty navigating EU...,[A streamlined and science-responsive risk ass...,[The EU biotech investment landscape is domina...,[While the EU is a leader in biologics manufac...,[],[],"[However, many biotech SMEs lack access to the...",[The biotech revolution depends increasingly o...,[Fund EU-wide talent fellowships for bioengine...,[While the EU is a leader in biologics manufac...,[]
1,F3553888,cpme.2025-092.FINAL.Statement.Biotech.Act,[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[]
2,F3554271,Koppert submission for the biotech act consult...,[],[],[],[],[],"[Despite its proven benefits, biocontrol remai...",[Immediate simplification measures should be i...,[At this point we deem important for the EU to...,"[More specifically, the EU needs to create EU ...","[More specifically, the EU needs to create EU ...",[],[],"[Moreover, supporting public-private partnersh...",[Biotech clusters need to be significant and m...,[],[]
3,F3559517,EBSA Position Paper - OMNIBUS - 20250416 (1),[],[],[],[],[],[Examples include misaligned provisions in the...,"[With the upcoming Biotech Act, the European U...",[],[],[],[],[],[],"[Founded in 1996, the European Biosafety Assoc...","[Founded in 1996, the European Biosafety Assoc...",[]
4,F3561326,PhageEU Proposals for Amendments_Pharmaceutica...,[],[],[],[],[],"[This is crucial because, under current framew...",[The amendment thus aligns with the broader EU...,[],"[In particular, medicinal products based on ba...",[],[],[],[],[],[],[]


In [40]:
# assuming `results` and `df` have the same order and length
for i, r in enumerate(results):
    if i < len(df):
        r["reference"] = df.loc[i, "reference"]
        r["title"] = df.loc[i, "title"]


In [41]:
len(results)

149

In [42]:
results[:5]

[{'ATMP': ['Biotech firms report difficulty navigating EU regulatory environments, particularly for clinical trials, GMOs, and advanced therapies. (page 4)',
   'The European Biotech Act was introduced in response to these challenges and is expected to form the legislative backbone of a broader European life sciences strategy. It seeks to simplify complex regulatory pathways, streamline product development pipelines, and harmonize fragmented biotech policies across Member States. (page 4)'],
  'ATMP_gene_therapies': None,
  'ATMP_cell_therapies': None,
  'ATMP_tissue_engineered_products': None,
  'ATMP_personalised_medicine': None,
  'Access_to_Markets_hurdles': ['Biotech firms report difficulty navigating EU regulatory environments, particularly for clinical trials, GMOs, and advanced therapies. Regulatory divergence across Member States results in unnecessary costs, slow time-to-market, and underutilized research findings. (page 4)',
   'The EU biotech investment landscape is dominat

In [43]:
FINAL_JSON_PATH = Path(output_folder) / "brussels_final_results_with_ref_title.json"

try:
    # dump your list of dictionaries (results) as a single JSON array
    with open(FINAL_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"[DONE] Final merged results saved to {FINAL_JSON_PATH}")
except Exception as e:
    print(f"[ERROR] Failed to save final JSON: {e}")

[DONE] Final merged results saved to C:\Users\YağmurAslan\OneDrive - Technopolis Group Ltd\RAG_Brussels\results\brussels_final_results_with_ref_title.json


In [33]:
output_folder = Path(folder_path).parent / "results"
xlsx_path = Path(output_folder) / "RAG_pp_results_table.xlsx"

df.to_excel(xlsx_path, index=False)

## Feedbacks

In [68]:
# Load the Excel sheet, skip first 3 rows, treat the 4th as header
fb = pd.read_excel(
    os.path.join(folder_path, "Working file-biotech act call for evidence telethon_VP Check.xlsx"),
    sheet_name="Sector analysis",
    skiprows=3
)
fb.head()

,Reference publication,Feedback reference,Ares reference,Date feedback,User type,Tr _ number,Organization,Sector - from EC,Scope,Governance level,...,title,organisation,page_number,summary,main themes,VC,clusters,skills,AI/data,dual_use/security
0,Ares(2025)3876590,F3551675,Ares(2025)3913571,2025/05/15 10:17:58,Business Association,NaN,COMPAG,Medical/pharma,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Ares(2025)3876590,F3551693,Ares(2025)3918507,2025/05/15 13:44:22,EU Citizen,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Ares(2025)3876590,F3551766,Ares(2025)3957144,2025/05/16 12:02:14,EU Citizen,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Ares(2025)3876590,F3551820,Ares(2025)3978576,2025/05/17 04:02:15,EU Citizen,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Ares(2025)3876590,F3551923,Ares(2025)3990974,2025/05/18 16:26:28,NGO (Non-governmental organisation),629482044225-60,OGM dangers,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Send texts to the blob container

In [45]:
from azure.storage.blob import BlobServiceClient
import os

# --- Connect to Azure Blob Storage ---
blob_service = BlobServiceClient.from_connection_string(os.environ["AZURE_BLOB_CONNECTION_STRING"])
container_client = blob_service.get_container_client("sante-docs")

# --- Loop through rows of fb and upload each feedback ---
for _, row in fb.iterrows():
    reference = str(row["Feedback reference"]).strip()
    text = str(row["Feedback   in   original   language"]).strip()

    # Skip rows with missing values
    if not reference or not text or reference.lower() == "nan":
        continue

    # Construct blob name using the reference
    blob_name = f"{reference}.txt"

    # Upload the feedback as a separate text file
    blob_client = container_client.get_blob_client(blob_name)
    blob_client.upload_blob(text.encode("utf-8"), overwrite=True)

    print(f"✅ Uploaded: {blob_name}")


✅ Uploaded: F3552803.txt
✅ Uploaded: F3553888.txt
✅ Uploaded: F3554271.txt
✅ Uploaded: F3559517.txt
✅ Uploaded: F3561326.txt
✅ Uploaded: F3563493.txt
✅ Uploaded: F3563646.txt
✅ Uploaded: F3563652.txt
✅ Uploaded: F3563668.txt
✅ Uploaded: F3563901.txt
✅ Uploaded: F3563910.txt
✅ Uploaded: F3563929.txt
✅ Uploaded: F3564074.txt
✅ Uploaded: F3564102.txt
✅ Uploaded: F3564121.txt
✅ Uploaded: F3564204.txt
✅ Uploaded: F3564210.txt
✅ Uploaded: F3564218.txt
✅ Uploaded: F3564247.txt
✅ Uploaded: F3564250.txt
✅ Uploaded: F3564253.txt
✅ Uploaded: F3564264.txt
✅ Uploaded: F3564266.txt
✅ Uploaded: F3564268.txt
✅ Uploaded: F3564270.txt
✅ Uploaded: F3564380.txt
✅ Uploaded: F3564435.txt
✅ Uploaded: F3564477.txt
✅ Uploaded: F3564478.txt
✅ Uploaded: F3564488.txt
✅ Uploaded: F3564504.txt
✅ Uploaded: F3564682.txt
✅ Uploaded: F3564687.txt
✅ Uploaded: F3564728.txt
✅ Uploaded: F3564760.txt
✅ Uploaded: F3564769.txt
✅ Uploaded: F3564784.txt
✅ Uploaded: F3564813.txt
✅ Uploaded: F3564816.txt
✅ Uploaded: F3564818.txt


In [46]:
len(fb)

222

In [47]:
container_name="sante-docs"
colect = UserDocumentCollection(container_name)

colect.reset_and_run_indexer()

2025-11-03 15:32:23.825 | INFO     | embeddings:get_container:66 - Container 'sante-docs' already exists.
2025-11-03 15:32:26.311 | INFO     | embeddings:reset_and_run_indexer:252 - 🧹 Indexer 'sante-docs-indexer' checkpoint reset successfully.
2025-11-03 15:32:26.543 | INFO     | embeddings:reset_and_run_indexer:256 - 🚀 Indexer 'sante-docs-indexer' started successfully.


In [48]:
from tqdm.auto import tqdm
import os, json, time

# --- Parameters ---
SAVE_EVERY = 10
OUTPUT_PATH = Path(output_folder)/"feedback_results.json"

# --- Get only uploaded .txt feedback files ---
docs = sorted([blob.name for blob in container_client.list_blobs() if blob.name.endswith(".txt")])

results, errors = [], []
pbar = tqdm(docs, desc="Processing feedback documents from blob", unit="doc", total=len(docs))

def save_checkpoint():
    """Save current progress to disk."""
    try:
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        temp_file = f"{os.path.splitext(OUTPUT_PATH)[0]}_checkpoint_{timestamp}.json"
        with open(temp_file, "w", encoding="utf-8") as f:
            json.dump({"results": results, "errors": errors}, f, ensure_ascii=False, indent=2)
        pbar.write(f"[SAVED] checkpoint with {len(results)} results → {temp_file}")
    except Exception as e:
        pbar.write(f"[WARNING] Failed to save checkpoint: {e}")

for i, doc in enumerate(pbar, start=1):
    pbar.set_postfix_str(doc)
    try:
        first  = call_once_with_wait(first_information,  doc, delay=10, label=f"first:{doc}")
        second = call_once_with_wait(second_information, doc, delay=10, label=f"second:{doc}")
        third  = call_once_with_wait(third_information,  doc, delay=10, label=f"third:{doc}")
        results.append({**first, **second, **third})
    except Exception as e:
        errors.append((doc, repr(e)))
        pbar.write(f"[ERROR] {doc}: {e}")

    # periodic checkpoint
    if i % SAVE_EVERY == 0:
        save_checkpoint()

# --- Final save ---
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump({"results": results, "errors": errors}, f, ensure_ascii=False, indent=2)
pbar.write(f"[DONE] All results saved to {OUTPUT_PATH}")


Processing feedback documents from blob:   0%|          | 0/222 [00:00<?, ?doc/s, F3551675.txt]

15:34:57 [first:F3551675.txt] waiting 10s before call...
15:35:09 [first:F3551675.txt] call OK
15:35:09 [second:F3551675.txt] waiting 10s before call...
15:35:21 [second:F3551675.txt] call OK
15:35:21 [third:F3551675.txt] waiting 10s before call...
15:35:34 [third:F3551675.txt] call OK


Processing feedback documents from blob:   0%|          | 1/222 [00:37<2:17:40, 37.38s/doc, F3551693.txt]

15:35:34 [first:F3551693.txt] waiting 10s before call...
15:35:46 [first:F3551693.txt] call OK
15:35:46 [second:F3551693.txt] waiting 10s before call...
15:35:59 [second:F3551693.txt] call OK
15:35:59 [third:F3551693.txt] waiting 10s before call...
15:36:11 [third:F3551693.txt] call OK


Processing feedback documents from blob:   1%|          | 2/222 [01:14<2:17:09, 37.40s/doc, F3551766.txt]

...

[SAVED] checkpoint with 220 results → C:\Users\YağmurAslan\OneDrive - Technopolis Group Ltd\RAG_Brussels\results\feedback_results_checkpoint_20251103_175924.json
17:59:24 [first:F3565532.txt] waiting 10s before call...
17:59:38 [first:F3565532.txt] call OK
17:59:38 [second:F3565532.txt] waiting 10s before call...
17:59:52 [second:F3565532.txt] call OK
17:59:52 [third:F3565532.txt] waiting 10s before call...
18:00:05 [third:F3565532.txt] call OK


Processing feedback documents from blob: 100%|█████████▉| 221/222 [2:25:08<00:39, 39.84s/doc, F3582124.txt]

18:00:05 [first:F3582124.txt] waiting 10s before call...
18:00:23 [first:F3582124.txt] call OK
18:00:23 [second:F3582124.txt] waiting 10s before call...
18:00:37 [second:F3582124.txt] call OK
18:00:37 [third:F3582124.txt] waiting 10s before call...
18:00:50 [third:F3582124.txt] call OK


Processing feedback documents from blob: 100%|██████████| 222/222 [2:25:53<00:00, 39.43s/doc, F3582124.txt]

[DONE] All results saved to C:\Users\YağmurAslan\OneDrive - Technopolis Group Ltd\RAG_Brussels\results\feedback_results.json


In [59]:
results

[{'ATMP': None,
  'ATMP_gene_therapies': None,
  'ATMP_cell_therapies': None,
  'ATMP_tissue_engineered_products': None,
  'ATMP_personalised_medicine': None,
  'Access_to_Markets_hurdles': None,
  'Access_to_Markets_strategies': None,
  'VC_Investments_IPO': None,
  'Manufacturing_Capabilities': None,
  'Incubation_and_Acceleration': None,
  'Talent_Retention': None,
  'Upskilling_Reskilling': None,
  'AI_Challenges': None,
  'AI_Development_and_Deployment': None,
  'Global_Collaboration': None,
  'Biosecurity': ["Non si può trascurare che va ad interessare dei punti di criticità da un punto di vista etico e della sicurezza della salute dei cittadini e pertanto necessita di un'attenta valutazione. (page 1)"],
  'Biosecurity_Nucleic_Acid_Screening': None,
  'reference': 'F3552803'},
 {'ATMP': None,
  'ATMP_gene_therapies': None,
  'ATMP_cell_therapies': None,
  'ATMP_tissue_engineered_products': None,
  'ATMP_personalised_medicine': None,
  'Access_to_Markets_hurdles': None,
  'Access_

In [53]:
fb

,Reference publication,Feedback reference,Ares reference,Date feedback,User type,Tr _ number,Organization,Sector - from EC,Scope,Governance level,...,title,organisation,page_number,summary,main themes,VC,clusters,skills,AI/data,dual_use/security
0,Ares(2025)3876590,F3552803,Ares(2025)4046089,2025/05/18 22:52:52,NGO (Non-governmental organisation),NaN,RareGen Youth Network,General,NaN,NaN,...,Feedback on the European Biotech Act,RareGen Youth Network,12.0,This document provides feedback on the Europea...,"['Regulatory fragmentation and bottlenecks', '...",The document highlights persistent gaps in ris...,The document discusses the importance of biote...,A shortage of skilled biotech professionals is...,The document stresses the growing importance o...,The document raises concerns about dual-use ri...
1,Ares(2025)3876590,F3553888,Ares(2025)4046970,2025/05/20 11:30:18,NGO (Non-governmental organisation),9276943405-41,Standing Committee of European Doctors (CPME),Medical/pharma,NaN,NaN,...,Statement on the European Commission call for ...,Standing Committee of European Doctors (CPME),1.0,The document is a statement from the Standing ...,"['Biotech Act consultation', 'Transparency in ...",The document urges the European Commission to ...,NaN,NaN,NaN,NaN
2,Ares(2025)3876590,F3554271,Ares(2025)4099470,2025/05/21 09:59:36,Company/business,852199818450-61,Koppert B.V.,Agri,NaN,NaN,...,Koppert’s Contribution to the EU Biotech Act C...,Koppert B.V.,2.0,Koppert's submission to the EU Biotech Act con...,"['Biocontrol inclusion in EU Biotech Act', 'Re...",The document highlights the need for the EU to...,It advocates for the creation of EU innovation...,NaN,The document proposes supporting public–privat...,NaN
3,Ares(2025)3876590,F3559517,Ares(2025)4262845,2025/05/27 10:28:48,Other,194518991651-02,EBSA ivzw,Biosecurity,NaN,NaN,...,Maturing Biological Risk Management for EU com...,EBSA ivzw,12.0,This position paper by the European Biosafety ...,"['Regulatory simplification in life sciences',...",NaN,NaN,NaN,NaN,The document highlights the need for a unified...
4,Ares(2025)3876590,F3561326,Ares(2025)4354009,2025/05/28 16:07:11,NGO (Non-governmental organisation),174779050296-36,PhageEU,Medical/pharma,NaN,NaN,...,PhageEU proposals for amendments to Pharmaceut...,PhageEU,7.0,This document proposes amendments to the EU Ph...,['Integration of phage therapies into EU pharm...,The document highlights that explicit inclusio...,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
217,Ares(2025)3876590,F3565258,Ares(2025)4656034,2025/06/11 14:59:28,Public authority,NaN,Danish National Center for Ethics,General,International,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
218,Ares(2025)3876590,F3564863,Ares(2025)4633359,2025/06/10 18:46:47,Public authority,560693115075-38,AGES,Medical/pharma,National,Agency,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
219,Ares(2025)3876590,F3565357,Ares(2025)4660574,2025/06/11 17:19:09,Public authority,NaN,Secrétariat général des Affaires européennes,"Industrial, food and feed, medical/pharma",National,Authority,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
220,Ares(2025)3876590,F3564832,Ares(2025)4629830,2025/06/10 16:56:32,Public authority,NaN,National Institute for Public Health and the E...,NaN,National,Agency,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [54]:
len(results)

222

In [55]:
fb.columns

Index(['Reference publication', 'Feedback reference', 'Ares reference',
       'Date feedback', 'User type', 'Tr _ number', 'Organization',
       'Sector - from EC', 'Scope', 'Governance level', 'Company size',
       'Country', 'First name', 'Surname', 'Email', 'Publication status',
       'Feedback language', 'Feedback   in   original   language',
       'Feedback   english   translation', 'Attachment', 'Status', 'Reviewed',
       'Login', 'First report date', 'Has to be reviewed',
       'papers to publish anonymised', 'file_name', 'Doc title _raw',
       'to anonymize', 'Stakeholder group', 'Country.1', 'title',
       'organisation', 'page_number', 'summary', 'main themes', 'VC',
       'clusters', 'skills', 'AI/data', 'dual_use/security'],
      dtype='object')

In [77]:
sorted_refs = fb["Feedback reference"].sort_values().tolist()

# Step 2: Assign these sorted references to your results
for i, r in enumerate(results):
    if i < len(sorted_refs):
        r["reference"] = sorted_refs[i]
    

In [78]:
results[:5]

[{'ATMP': None,
  'ATMP_gene_therapies': None,
  'ATMP_cell_therapies': None,
  'ATMP_tissue_engineered_products': None,
  'ATMP_personalised_medicine': None,
  'Access_to_Markets_hurdles': None,
  'Access_to_Markets_strategies': None,
  'VC_Investments_IPO': None,
  'Manufacturing_Capabilities': None,
  'Incubation_and_Acceleration': None,
  'Talent_Retention': None,
  'Upskilling_Reskilling': None,
  'AI_Challenges': None,
  'AI_Development_and_Deployment': None,
  'Global_Collaboration': None,
  'Biosecurity': ["Non si può trascurare che va ad interessare dei punti di criticità da un punto di vista etico e della sicurezza della salute dei cittadini e pertanto necessita di un'attenta valutazione. (page 1)"],
  'Biosecurity_Nucleic_Acid_Screening': None,
  'reference': 'F3551675'},
 {'ATMP': None,
  'ATMP_gene_therapies': None,
  'ATMP_cell_therapies': None,
  'ATMP_tissue_engineered_products': None,
  'ATMP_personalised_medicine': None,
  'Access_to_Markets_hurdles': None,
  'Access_

In [79]:
# Let's assume 'results' is your updated list of dictionaries
with open('brussels_fb_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

### Replace pages with feedback

In [83]:
import re

def replace_page_with_feedback(lst):
    if isinstance(lst, list):
        return [
            re.sub(r"\(page\s*\d+\)", "(feedback)", s) if isinstance(s, str) else s
            for s in lst
        ]
    return lst

for d in results:
    for k, v in d.items():
        if isinstance(v, list):
            d[k] = replace_page_with_feedback(v)


In [84]:
results[:5]

[{'ATMP': None,
  'ATMP_gene_therapies': None,
  'ATMP_cell_therapies': None,
  'ATMP_tissue_engineered_products': None,
  'ATMP_personalised_medicine': None,
  'Access_to_Markets_hurdles': None,
  'Access_to_Markets_strategies': None,
  'VC_Investments_IPO': None,
  'Manufacturing_Capabilities': None,
  'Incubation_and_Acceleration': None,
  'Talent_Retention': None,
  'Upskilling_Reskilling': None,
  'AI_Challenges': None,
  'AI_Development_and_Deployment': None,
  'Global_Collaboration': None,
  'Biosecurity': ["Non si può trascurare che va ad interessare dei punti di criticità da un punto di vista etico e della sicurezza della salute dei cittadini e pertanto necessita di un'attenta valutazione. (feedback)"],
  'Biosecurity_Nucleic_Acid_Screening': None,
  'reference': 'F3551675'},
 {'ATMP': None,
  'ATMP_gene_therapies': None,
  'ATMP_cell_therapies': None,
  'ATMP_tissue_engineered_products': None,
  'ATMP_personalised_medicine': None,
  'Access_to_Markets_hurdles': None,
  'Acces

In [85]:
# Let's assume 'results' is your updated list of dictionaries
with open(Path(output_folder) /'brussels_feedback_replaced_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)


## MERGE PP AND FB

In [94]:
import json
from pathlib import Path



with (output_folder / 'brussels_final_results_with_ref_title.json').open(encoding='utf-8') as f:
    pp = json.load(f)

with (output_folder / 'brussels_feedback_replaced_results.json').open(encoding='utf-8') as f:
    fb = json.load(f)

# Build a lookup for pp by reference
pp_refs = {d['reference']: d for d in pp if 'reference' in d}

for fb_dict in fb:
    ref = fb_dict.get('reference')
    if ref in pp_refs:
        pp_dict = pp_refs[ref]
        for k, v in fb_dict.items():
            if k == "reference":
                continue
            # Merge lists, else set/overwrite
            if v is not None:
                if k in pp_dict and isinstance(pp_dict[k], list) and isinstance(v, list):
                    pp_dict[k].extend(v)
                else:
                    pp_dict[k] = v
    else:
        pp.append(fb_dict)

# Save merged result
with (output_folder / 'brussels_merged_results.json').open('w', encoding='utf-8') as f:
    json.dump(pp, f, ensure_ascii=False, indent=2)



In [97]:
import pandas as pd

# Convert list of dicts to DataFrame
df = pd.DataFrame(pp)

# Make sure both 'reference' and 'title' are present
columns = df.columns.tolist()
first_cols = [col for col in ['reference', 'title'] if col in columns]
other_cols = [col for col in columns if col not in first_cols]
ordered_cols = first_cols + other_cols

# Reorder
df = df[ordered_cols]


# Save to Excel
df.to_excel(Path(output_folder) / 'brussels_final_merged_results.xlsx', index=False)

## NON ENG FEEDBACK TRANSLATE

In [17]:
# Load the Excel sheet, skip first 3 rows, treat the 4th as header
fb = pd.read_excel(
    os.path.join(folder_path, "Working file-biotech act call for evidence telethon_VP Check.xlsx"),
    sheet_name="Sector analysis",
    skiprows=3
)
fb.head()

,Reference publication,Feedback reference,Ares reference,Date feedback,User type,Tr _ number,Organization,Sector - from EC,Scope,Governance level,...,title,organisation,page_number,summary,main themes,VC,clusters,skills,AI/data,dual_use/security
0,Ares(2025)3876590,F3551675,Ares(2025)3913571,2025/05/15 10:17:58,Business Association,NaN,COMPAG,Medical/pharma,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Ares(2025)3876590,F3551693,Ares(2025)3918507,2025/05/15 13:44:22,EU Citizen,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Ares(2025)3876590,F3551766,Ares(2025)3957144,2025/05/16 12:02:14,EU Citizen,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Ares(2025)3876590,F3551820,Ares(2025)3978576,2025/05/17 04:02:15,EU Citizen,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Ares(2025)3876590,F3551923,Ares(2025)3990974,2025/05/18 16:26:28,NGO (Non-governmental organisation),629482044225-60,OGM dangers,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
fb.columns

Index(['Reference publication', 'Feedback reference', 'Ares reference',
       'Date feedback', 'User type', 'Tr _ number', 'Organization',
       'Sector - from EC', 'Scope', 'Governance level', 'Company size',
       'Country', 'First name', 'Surname', 'Email', 'Publication status',
       'Feedback language', 'Feedback   in   original   language',
       'Feedback   english   translation', 'Attachment', 'Status', 'Reviewed',
       'Login', 'First report date', 'Has to be reviewed',
       'papers to publish anonymised', 'file_name', 'Doc title _raw',
       'to anonymize', 'Stakeholder group', 'Country.1', 'title',
       'organisation', 'page_number', 'summary', 'main themes', 'VC',
       'clusters', 'skills', 'AI/data', 'dual_use/security'],
      dtype='object')

In [19]:
# --- Filter rows with a value in "Feedback   english   translation" ---
mask = fb["Feedback   english   translation"].notnull() & (fb["Feedback   english   translation"].astype(str).str.strip() != "")
filtered_fb = fb[mask]


# Print the number of rows with a translation
print(len(filtered_fb))

34


In [20]:
from azure.storage.blob import BlobServiceClient
import os

# --- Connect to Azure Blob Storage ---
blob_service = BlobServiceClient.from_connection_string(os.environ["AZURE_BLOB_CONNECTION_STRING"])
container_client = blob_service.get_container_client("sante-docs")


# --- Upload each translation as a separate text file ---
for _, row in filtered_fb.iterrows():
    reference = str(row["Feedback reference"]).strip()
    text = str(row["Feedback   english   translation"]).strip()

    # Construct blob name
    blob_name = f"{reference}_en.txt"

    # Upload
    blob_client = container_client.get_blob_client(blob_name)
    blob_client.upload_blob(text.encode("utf-8"), overwrite=True)

    print(f"✅ Uploaded: {blob_name}")


✅ Uploaded: F3551675_en.txt
✅ Uploaded: F3551693_en.txt
✅ Uploaded: F3551766_en.txt
✅ Uploaded: F3551820_en.txt
✅ Uploaded: F3551923_en.txt
✅ Uploaded: F3553253_en.txt
✅ Uploaded: F3553267_en.txt
✅ Uploaded: F3553780_en.txt
✅ Uploaded: F3562778_en.txt
✅ Uploaded: F3563249_en.txt
✅ Uploaded: F3563345_en.txt
✅ Uploaded: F3563368_en.txt
✅ Uploaded: F3563451_en.txt
✅ Uploaded: F3563646_en.txt
✅ Uploaded: F3564253_en.txt
✅ Uploaded: F3564264_en.txt
✅ Uploaded: F3564435_en.txt
✅ Uploaded: F3564641_en.txt
✅ Uploaded: F3564816_en.txt
✅ Uploaded: F3564900_en.txt
✅ Uploaded: F3564967_en.txt
✅ Uploaded: F3565034_en.txt
✅ Uploaded: F3565059_en.txt
✅ Uploaded: F3565157_en.txt
✅ Uploaded: F3565164_en.txt
✅ Uploaded: F3565196_en.txt
✅ Uploaded: F3565357_en.txt
✅ Uploaded: F3565369_en.txt
✅ Uploaded: F3565421_en.txt
✅ Uploaded: F3565446_en.txt
✅ Uploaded: F3565491_en.txt
✅ Uploaded: F3565493_en.txt
✅ Uploaded: F3565495_en.txt
✅ Uploaded: F3565520_en.txt


In [21]:
container_name="sante-docs"
colect = UserDocumentCollection(container_name)

colect.reset_and_run_indexer()

2025-11-04 11:13:13.432 | INFO     | embeddings:get_container:66 - Container 'sante-docs' already exists.
2025-11-04 11:13:24.996 | INFO     | embeddings:reset_and_run_indexer:252 - 🧹 Indexer 'sante-docs-indexer' checkpoint reset successfully.
2025-11-04 11:13:25.237 | INFO     | embeddings:reset_and_run_indexer:256 - 🚀 Indexer 'sante-docs-indexer' started successfully.


In [24]:
from tqdm.auto import tqdm
import os, json, time

# --- Parameters ---
SAVE_EVERY = 10
OUTPUT_PATH = Path(output_folder)/"feedback_translation_results.json"

# --- Get only uploaded .txt feedback files ---
docs = sorted([blob.name for blob in container_client.list_blobs() if blob.name.endswith(".txt")])

results, errors = [], []
pbar = tqdm(docs, desc="Processing feedback documents from blob", unit="doc", total=len(docs))

def save_checkpoint():
    """Save current progress to disk."""
    try:
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        temp_file = f"{os.path.splitext(OUTPUT_PATH)[0]}_checkpoint_{timestamp}.json"
        with open(temp_file, "w", encoding="utf-8") as f:
            json.dump({"results": results, "errors": errors}, f, ensure_ascii=False, indent=2)
        pbar.write(f"[SAVED] checkpoint with {len(results)} results → {temp_file}")
    except Exception as e:
        pbar.write(f"[WARNING] Failed to save checkpoint: {e}")

for i, doc in enumerate(pbar, start=1):
    pbar.set_postfix_str(doc)
    try:
        first  = call_once_with_wait(first_information,  doc, delay=10, label=f"first:{doc}")
        second = call_once_with_wait(second_information, doc, delay=10, label=f"second:{doc}")
        third  = call_once_with_wait(third_information,  doc, delay=10, label=f"third:{doc}")
        results.append({**first, **second, **third})
    except Exception as e:
        errors.append((doc, repr(e)))
        pbar.write(f"[ERROR] {doc}: {e}")

    # periodic checkpoint
    if i % SAVE_EVERY == 0:
        save_checkpoint()

# --- Final save ---
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump({"results": results, "errors": errors}, f, ensure_ascii=False, indent=2)
pbar.write(f"[DONE] All results saved to {OUTPUT_PATH}")


Processing feedback documents from blob:   0%|          | 0/34 [00:00<?, ?doc/s, F3551675_en.txt]

11:14:51 [first:F3551675_en.txt] waiting 10s before call...
11:15:06 [first:F3551675_en.txt] call OK
11:15:06 [second:F3551675_en.txt] waiting 10s before call...
11:15:20 [second:F3551675_en.txt] call OK
11:15:20 [third:F3551675_en.txt] waiting 10s before call...
11:15:32 [third:F3551675_en.txt] call OK


Processing feedback documents from blob:   3%|▎         | 1/34 [00:41<22:47, 41.45s/doc, F3551693_en.txt]

11:15:32 [first:F3551693_en.txt] waiting 10s before call...
11:15:46 [first:F3551693_en.txt] call OK
11:15:46 [second:F3551693_en.txt] waiting 10s before call...
11:15:59 [second:F3551693_en.txt] call OK
11:15:59 [third:F3551693_en.txt] waiting 10s before call...
11:16:11 [third:F3551693_en.txt] call OK


Processing feedback documents from blob:   6%|▌         | 2/34 [01:20<21:22, 40.07s/doc, F3551766_en.txt]

...

11:36:30 [first:F3565495_en.txt] waiting 10s before call...
11:36:43 [first:F3565495_en.txt] call OK
11:36:43 [second:F3565495_en.txt] waiting 10s before call...
11:36:56 [second:F3565495_en.txt] call OK
11:36:56 [third:F3565495_en.txt] waiting 10s before call...
11:37:09 [third:F3565495_en.txt] call OK


Processing feedback documents from blob:  97%|█████████▋| 33/34 [22:17<00:41, 41.05s/doc, F3565520_en.txt]

11:37:09 [first:F3565520_en.txt] waiting 10s before call...
11:37:23 [first:F3565520_en.txt] call OK
11:37:23 [second:F3565520_en.txt] waiting 10s before call...
11:37:35 [second:F3565520_en.txt] call OK
11:37:35 [third:F3565520_en.txt] waiting 10s before call...
11:37:47 [third:F3565520_en.txt] call OK


Processing feedback documents from blob: 100%|██████████| 34/34 [22:56<00:00, 40.49s/doc, F3565520_en.txt]

[DONE] All results saved to C:\Users\YağmurAslan\OneDrive - Technopolis Group Ltd\RAG_Brussels\results\feedback_translation_results.json


In [28]:
results[:3]

[{'ATMP': None,
  'ATMP_gene_therapies': None,
  'ATMP_cell_therapies': None,
  'ATMP_tissue_engineered_products': None,
  'ATMP_personalised_medicine': None,
  'Access_to_Markets_hurdles': ['It cannot be overlooked that it concerns critical points of concern from an ethical point of view and the safety of citizens’ health and therefore requires careful assessment. (page 1)',
   'An assessment to overcome vetoes that are often placed under the precautionary principle often used ideologically (page 1)'],
  'Access_to_Markets_strategies': None,
  'VC_Investments_IPO': None,
  'Manufacturing_Capabilities': None,
  'Incubation_and_Acceleration': None,
  'Talent_Retention': None,
  'Upskilling_Reskilling': None,
  'AI_Challenges': None,
  'AI_Development_and_Deployment': None,
  'Global_Collaboration': None,
  'Biosecurity': None,
  'Biosecurity_Nucleic_Acid_Screening': None,
  'reference': 'F3551675'},
 {'ATMP': None,
  'ATMP_gene_therapies': None,
  'ATMP_cell_therapies': None,
  'ATMP_ti

In [26]:
len(results)

34

In [27]:
# Sort the 'Feedback reference' of these filtered rows
sorted_refs = filtered_fb["Feedback reference"].sort_values().tolist()

# Assign these sorted references to your results
for i, r in enumerate(results):
    if i < len(sorted_refs):
        r["reference"] = sorted_refs[i]

In [29]:
# Let's assume 'results' is your updated list of dictionaries
with open(Path(output_folder)/'brussels_fb_translation_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

In [30]:
import re

def replace_page_with_feedback(lst):
    if isinstance(lst, list):
        return [
            re.sub(r"\(page\s*\d+\)", "(from feedback translation)", s) if isinstance(s, str) else s
            for s in lst
        ]
    return lst

for d in results:
    for k, v in d.items():
        if isinstance(v, list):
            d[k] = replace_page_with_feedback(v)

In [32]:
# Let's assume 'results' is your updated list of dictionaries
with open(Path(output_folder)/'brussels_fb_translation_page_replaced.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

In [33]:
results[:5]

[{'ATMP': None,
  'ATMP_gene_therapies': None,
  'ATMP_cell_therapies': None,
  'ATMP_tissue_engineered_products': None,
  'ATMP_personalised_medicine': None,
  'Access_to_Markets_hurdles': ['It cannot be overlooked that it concerns critical points of concern from an ethical point of view and the safety of citizens’ health and therefore requires careful assessment. (from feedback translation)',
   'An assessment to overcome vetoes that are often placed under the precautionary principle often used ideologically (from feedback translation)'],
  'Access_to_Markets_strategies': None,
  'VC_Investments_IPO': None,
  'Manufacturing_Capabilities': None,
  'Incubation_and_Acceleration': None,
  'Talent_Retention': None,
  'Upskilling_Reskilling': None,
  'AI_Challenges': None,
  'AI_Development_and_Deployment': None,
  'Global_Collaboration': None,
  'Biosecurity': None,
  'Biosecurity_Nucleic_Acid_Screening': None,
  'reference': 'F3551675'},
 {'ATMP': None,
  'ATMP_gene_therapies': None,
  '

In [34]:
import json
from pathlib import Path



with (output_folder / 'brussels_merged_results.json').open(encoding='utf-8') as f:
    pp = json.load(f)

with (output_folder / 'brussels_fb_translation_page_replaced.json').open(encoding='utf-8') as f:
    fb = json.load(f)

# Build a lookup for pp by reference
pp_refs = {d['reference']: d for d in pp if 'reference' in d}

for fb_dict in fb:
    ref = fb_dict.get('reference')
    if ref in pp_refs:
        pp_dict = pp_refs[ref]
        for k, v in fb_dict.items():
            if k == "reference":
                continue
            # Merge lists, else set/overwrite
            if v is not None:
                if k in pp_dict and isinstance(pp_dict[k], list) and isinstance(v, list):
                    pp_dict[k].extend(v)
                else:
                    pp_dict[k] = v
    else:
        pp.append(fb_dict)

# Save merged result
with (output_folder / 'brussels_merged_results_translation_added.json').open('w', encoding='utf-8') as f:
    json.dump(pp, f, ensure_ascii=False, indent=2)

In [35]:
import pandas as pd

# Convert list of dicts to DataFrame
df = pd.DataFrame(pp)

# Make sure both 'reference' and 'title' are present
columns = df.columns.tolist()
first_cols = [col for col in ['reference', 'title'] if col in columns]
other_cols = [col for col in columns if col not in first_cols]
ordered_cols = first_cols + other_cols

# Reorder
df = df[ordered_cols]


# Save to Excel
df.to_excel(Path(output_folder) / 'brussels_final_merged_translation_added.xlsx', index=False)